# Objective Track: SigLIP 2 (age, gender, ethnicity)

Run order (Colab): run all sections from top to bottom. Section 4 scores the 2,880 images and can be stopped and resumed.

How the model "answers": for each image, SigLIP 2 scores one text prompt per option. The option with the highest score is its answer. It always picks an option, so there is no letter rule, no unclear-answer files, no matcher, and no "none" answers.

Decisions fixed in this notebook:
- Checkpoint: `google/siglip2-giant-opt-patch16-384` (chosen because it is the largest SigLIP 2 model, not because it was tested on faces).
- Prompts: the 4 templates of Hausladen et al. (Table 1), with each option inserted as the adjective. The score of an option is the **average over the 4 templates**. Averaging logits instead of cosine similarities is our adaptation, since Hausladen et al. average similarities to score traits, not to pick a class.
- Option wording, and the a/an agreement ("an African person"), are our own design choices (no source).
- Text is lowercased and padded to 64 tokens, as the SigLIP 2 documentation requires.
- McNemar's test is the **exact (binomial)** version, with **no multiple-comparison correction**.

## 1. Setup, paths, load data

In [1]:
# [section 1a: setup]
import os, re, json, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import binomtest

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# [section 1b: config]
MAIN_DIR = '/content/drive/MyDrive/Frugal AI Project/'
OUTPUT_DIR = os.path.join(MAIN_DIR, 'Objective Track')
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABELS_PATH = os.path.join(MAIN_DIR, 'sampled_dataset.csv')
BASE_DIR    = os.path.join(MAIN_DIR, 'generated_portraits_hashed_final/generated_portraits_hashed')
CAP_DIR     = os.path.join(MAIN_DIR, 'generated_cap')
GLASSES_DIR = os.path.join(MAIN_DIR, 'generated_glasses')
MASK_DIR    = os.path.join(MAIN_DIR, 'generated_masks')

IMAGE_TYPES = ['base', 'cap', 'glasses', 'mask']
ACCESSORIES = ['cap', 'glasses', 'mask']
ATTRIBUTES  = ['ethnicity', 'gender', 'age']

MODEL_ID = 'google/siglip2-giant-opt-patch16-384'
RAW_PATH = os.path.join(OUTPUT_DIR, 'siglip2_raw_scores.jsonl')
PROMPTS_PATH = os.path.join(OUTPUT_DIR, 'siglip2_prompts.json')

In [3]:
# [section 1c: load labels and check image files]
REQUIRED = ['HASH', 'ETHNICITY', 'GENDER', 'AGE']
df = pd.read_csv(LABELS_PATH)
assert set(REQUIRED) <= set(df.columns), f"missing columns, found {list(df.columns)}"
df = df[REQUIRED].copy()
assert df['HASH'].is_unique, "duplicate HASH values"
print("identities:", len(df))

def image_path(hash_, image_type):
    if image_type == 'base':    return f"{BASE_DIR}/{hash_}.jpg"
    if image_type == 'cap':     return f"{CAP_DIR}/{hash_}_hat.png"
    if image_type == 'glasses': return f"{GLASSES_DIR}/{hash_}_glasses.png"
    if image_type == 'mask':    return f"{MASK_DIR}/{hash_}_mask.png"
    raise ValueError(image_type)

missing = [(h, t) for h in df['HASH'] for t in IMAGE_TYPES if not os.path.exists(image_path(h, t))]
assert not missing, f"{len(missing)} image files not found, first ones: {missing[:3]}"
print("all", len(df) * len(IMAGE_TYPES), "image files found")

identities: 720
all 2880 image files found


## 2. Prompts
Each option is inserted as the adjective into the 4 templates of Hausladen et al. (Table 1): "A photo of a <adjective> person.", "A <adjective> person.", "This is a <adjective> person.", "Cropped face photo of a <adjective> person."
The article is "an" before a vowel ("an african person") and "a" otherwise, because the templates hard-code "a". All text is lowercased.

In [4]:
# [section 2: prompts]
OPTIONS = {
    'ethnicity': ['African', 'Asian', 'Caucasian', 'Indian'],
    'gender':    ['Female', 'Male'],
    'age':       ['25', '40', '55'],
}
LETTERS = 'ABCD'

# adjective that goes into the templates, per option (same order as OPTIONS)
ADJECTIVES = {
    'ethnicity': ['African', 'Asian', 'Caucasian', 'Indian'],
    'gender':    ['female', 'male'],
    'age':       ['25-year-old', '40-year-old', '55-year-old'],
}
# Hausladen et al., Table 1, written with an {art} slot for "a"/"an"
TEMPLATES = [
    "A photo of {art} {adj} person.",
    "{Art} {adj} person.",
    "This is {art} {adj} person.",
    "Cropped face photo of {art} {adj} person.",
]

def article(adj):
    return 'an' if adj[0].lower() in 'aeiou' else 'a'

PROMPTS = []
for attribute in ATTRIBUTES:
    assert len(ADJECTIVES[attribute]) == len(OPTIONS[attribute])
    for oi, (option, adj) in enumerate(zip(OPTIONS[attribute], ADJECTIVES[attribute])):
        art = article(adj)
        for ti, template in enumerate(TEMPLATES):
            text = template.format(art=art, Art=art.capitalize(), adj=adj).lower()
            PROMPTS.append({'attribute': attribute, 'option_index': oi, 'option': option,
                            'template_index': ti, 'text': text})

TEXTS = [p['text'] for p in PROMPTS]
PROMPT_INDEX = {}   # (attribute, option_index) -> positions in TEXTS
for k, p in enumerate(PROMPTS):
    PROMPT_INDEX.setdefault((p['attribute'], p['option_index']), []).append(k)
assert all(len(v) == len(TEMPLATES) for v in PROMPT_INDEX.values())

# the true labels in the data must all be among the options
for attribute in ATTRIBUTES:
    labels = set(df[attribute.upper()].astype(str))
    assert labels <= set(OPTIONS[attribute]), f"{attribute}: labels {labels} not all in options"

with open(PROMPTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(PROMPTS, f, indent=1, ensure_ascii=False)

print(len(TEXTS), "prompts")
for t in TEXTS:
    print(t)

36 prompts
a photo of an african person.
an african person.
this is an african person.
cropped face photo of an african person.
a photo of an asian person.
an asian person.
this is an asian person.
cropped face photo of an asian person.
a photo of a caucasian person.
a caucasian person.
this is a caucasian person.
cropped face photo of a caucasian person.
a photo of an indian person.
an indian person.
this is an indian person.
cropped face photo of an indian person.
a photo of a female person.
a female person.
this is a female person.
cropped face photo of a female person.
a photo of a male person.
a male person.
this is a male person.
cropped face photo of a male person.
a photo of a 25-year-old person.
a 25-year-old person.
this is a 25-year-old person.
cropped face photo of a 25-year-old person.
a photo of a 40-year-old person.
a 40-year-old person.
this is a 40-year-old person.
cropped face photo of a 40-year-old person.
a photo of a 55-year-old person.
a 55-year-old person.
this i

## 3. Model and the function that scores one image
`padding="max_length"` and `max_length=64`, the lowercased text, `AutoModel` and `logits_per_image` follow the SigLIP 2 documentation on Hugging Face. The model is loaded in the default precision, with no dtype argument. One image is scored per call, as in the documentation example.

In [5]:
# [section 3: model]
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModel

assert torch.cuda.is_available(), "no GPU found"
device = 'cuda'
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()

def score_image(path):
    # returns one logit per prompt in TEXTS, in the same order
    image = Image.open(path).convert('RGB')
    inputs = processor(text=TEXTS, images=image, padding="max_length", max_length=64,
                       return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits_per_image[0].float().cpu().tolist()

test = score_image(image_path(df['HASH'].iloc[0], 'base'))
assert len(test) == len(TEXTS), f"expected {len(TEXTS)} logits, got {len(test)}"
print("model loaded, one image scored:", len(test), "logits")

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/101k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

model loaded, one image scored: 36 logits


## 4. Score all images
Every image's raw logits are appended to `siglip2_raw_scores.jsonl` as soon as they are computed. If the run stops, run this cell again: finished images are skipped.

In [6]:
# [section 4: run]
def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

done = set()
if os.path.exists(RAW_PATH):
    done = {(r['hash'], r['image_type']) for r in load_jsonl(RAW_PATH)}
print("already done:", len(done), "of", len(df) * len(IMAGE_TYPES))

with open(RAW_PATH, 'a', encoding='utf-8') as f:
    for hash_ in tqdm(df['HASH'], desc="identities"):
        for image_type in IMAGE_TYPES:
            if (hash_, image_type) in done:
                continue
            logits = score_image(image_path(hash_, image_type))
            f.write(json.dumps({'hash': hash_, 'image_type': image_type, 'logits': logits}) + '\n')
            f.flush()

already done: 2880 of 2880


identities:   0%|          | 0/720 [00:00<?, ?it/s]

## 5. Predict and grade
For each image and attribute: average the logits of the 4 templates for each option, take the option with the highest average as the answer, and compare it with the true label. SigLIP 2's logits are the cosine similarity times one scale plus one bias, shared by all prompts (as far as I know), so averaging logits picks the same option as averaging cosine similarities would.

In [7]:
# [section 5: predict and grade]
raw = load_jsonl(RAW_PATH)
expected_rows = len(df) * len(IMAGE_TYPES)
assert len(raw) == expected_rows, f"expected {expected_rows} scored images, found {len(raw)}"
assert len({(r['hash'], r['image_type']) for r in raw}) == expected_rows, "duplicate scored images"

labels = df.set_index('HASH')
rows = []
for r in raw:
    logits = np.asarray(r['logits'], dtype=float)
    assert len(logits) == len(TEXTS)
    for attribute in ATTRIBUTES:
        means = [logits[PROMPT_INDEX[(attribute, oi)]].mean() for oi in range(len(OPTIONS[attribute]))]
        pred_index = int(np.argmax(means))
        true_label = str(labels.loc[r['hash'], attribute.upper()])
        true_index = OPTIONS[attribute].index(true_label)
        rows.append({
            'hash': r['hash'], 'attribute': attribute, 'image_type': r['image_type'],
            'pred_option': OPTIONS[attribute][pred_index], 'pred_letter': LETTERS[pred_index],
            'true_label': true_label, 'true_letter': LETTERS[true_index],
            'correct': pred_index == true_index,
        })

graded = pd.DataFrame(rows)
graded.to_csv(os.path.join(OUTPUT_DIR, 'siglip2_graded_results.csv'), index=False)
print(len(graded), "graded answers")
graded.head()

8640 graded answers


,hash,attribute,image_type,pred_option,pred_letter,true_label,true_letter,correct
0,09534c3a20eaad55,ethnicity,base,African,A,African,A,True
1,09534c3a20eaad55,gender,base,Female,A,Female,A,True
2,09534c3a20eaad55,age,base,40,B,25,A,False
3,09534c3a20eaad55,ethnicity,cap,African,A,African,A,True
4,09534c3a20eaad55,gender,cap,Female,A,Female,A,True


## 6. Check: base accuracy
Base accuracy per attribute, the reference for every accessory result. Unclear and none rates do not exist for SigLIP 2, since it always picks an option.

In [8]:
# [section 6: checks]
base_acc = (graded[graded['image_type'] == 'base']
            .groupby('attribute')['correct'].agg(accuracy='mean', n='size').reindex(ATTRIBUTES))
base_acc['accuracy'] = (base_acc['accuracy'] * 100).round(2)
print("Base accuracy (%):"); print(base_acc.to_string())
base_acc.to_csv(os.path.join(OUTPUT_DIR, 'siglip2_check_base_accuracy.csv'))

Base accuracy (%):
           accuracy    n
attribute               
ethnicity     92.22  720
gender        99.58  720
age           55.69  720


## 7. RQ1: exact McNemar test per attribute x accessory
For each identity, compare correct/wrong on the base image with correct/wrong on the accessory image.
- b = correct on base, wrong with accessory.
- c = wrong on base, correct with accessory.
- Accuracy change = accuracy with accessory minus base accuracy = (c - b) / n.
- p-value: exact two-sided binomial test of b out of (b + c) against 0.5. If b + c = 0 there is no flipped identity and the p-value is NaN.

The 9 tests (3 attributes x 3 accessories) are not corrected for multiple comparisons.

In [9]:
# [section 7: McNemar]
def mcnemar_exact(base_correct, accessory_correct):
    base_correct = np.asarray(base_correct, dtype=bool)
    accessory_correct = np.asarray(accessory_correct, dtype=bool)
    n = len(base_correct)
    b = int((base_correct & ~accessory_correct).sum())    # correct -> wrong
    c = int((~base_correct & accessory_correct).sum())    # wrong -> correct
    p = binomtest(b, b + c, 0.5).pvalue if (b + c) > 0 else float('nan')
    return {
        'n': n,
        'acc_base': round(base_correct.mean() * 100, 2),
        'acc_accessory': round(accessory_correct.mean() * 100, 2),
        'b_correct_to_wrong': b,
        'c_wrong_to_correct': c,
        'acc_change_pts': round((c - b) / n * 100, 2),
        'p_value': p,
    }

def paired_results(graded, hashes=None):
    rows = []
    for attribute in ATTRIBUTES:
        g = graded[graded['attribute'] == attribute]
        if hashes is not None:
            g = g[g['hash'].isin(hashes)]
        wide = g.pivot(index='hash', columns='image_type', values='correct').astype(bool)
        for accessory in ACCESSORIES:
            rows.append({'attribute': attribute, 'accessory': accessory,
                         **mcnemar_exact(wide['base'].values, wide[accessory].values)})
    return pd.DataFrame(rows)

overall = paired_results(graded)
overall.to_csv(os.path.join(OUTPUT_DIR, 'siglip2_mcnemar_overall.csv'), index=False)
overall

,attribute,accessory,n,acc_base,acc_accessory,b_correct_to_wrong,c_wrong_to_correct,acc_change_pts,p_value
0,ethnicity,cap,720,92.22,87.92,33,2,-4.31,3.672903e-08
1,ethnicity,glasses,720,92.22,89.31,25,4,-2.92,1.037158e-04
2,ethnicity,mask,720,92.22,87.78,38,6,-4.44,9.430375e-07
3,gender,cap,720,99.58,95.00,33,0,-4.58,2.328306e-10
4,gender,glasses,720,99.58,98.75,6,0,-0.83,3.125000e-02
5,gender,mask,720,99.58,98.47,8,0,-1.11,7.812500e-03
6,age,cap,720,55.69,50.00,57,16,-5.69,1.525658e-06
7,age,glasses,720,55.69,53.19,48,30,-2.50,5.354439e-02
8,age,mask,720,55.69,38.06,128,1,-17.64,3.820357e-37


## 8. RQ3: the same test split by ethnicity, gender and age
`n` is the number of identities in the group. The tests are not corrected for multiple comparisons.

In [10]:
# [section 8: groups]
parts = []
for group_var in ['ETHNICITY', 'GENDER', 'AGE']:
    for value in sorted(df[group_var].unique()):
        hashes = set(df.loc[df[group_var] == value, 'HASH'])
        t = paired_results(graded, hashes)
        t.insert(0, 'group_var', group_var)
        t.insert(1, 'group', value)
        parts.append(t)

by_group = pd.concat(parts, ignore_index=True)
by_group.to_csv(os.path.join(OUTPUT_DIR, 'siglip2_mcnemar_by_group.csv'), index=False)
by_group

,group_var,group,attribute,accessory,n,acc_base,acc_accessory,b_correct_to_wrong,c_wrong_to_correct,acc_change_pts,p_value
0,ETHNICITY,African,ethnicity,cap,180,89.44,86.67,7,2,-2.78,1.796875e-01
1,ETHNICITY,African,ethnicity,glasses,180,89.44,87.22,6,2,-2.22,2.890625e-01
2,ETHNICITY,African,ethnicity,mask,180,89.44,87.78,5,2,-1.67,4.531250e-01
3,ETHNICITY,African,gender,cap,180,99.44,93.33,11,0,-6.11,9.765625e-04
4,ETHNICITY,African,gender,glasses,180,99.44,97.22,4,0,-2.22,1.250000e-01
...,...,...,...,...,...,...,...,...,...,...,...
76,AGE,55,gender,glasses,240,99.17,98.75,1,0,-0.42,1.000000e+00
77,AGE,55,gender,mask,240,99.17,97.08,5,0,-2.08,6.250000e-02
78,AGE,55,age,cap,240,65.83,49.17,53,13,-16.67,7.237175e-07
79,AGE,55,age,glasses,240,65.83,54.17,43,15,-11.67,3.069470e-04
